In [1]:
!pip install pyarrow
!pip install rank-bm25
!pip install beautifulsoup4
!pip install pymorphy3
!pip install sentence-transformers

#Описание решения
##Обработка HTML
 BeautifulSoup и re, извлечение текста с разделением пробелами, удаление скриптов и стилей, лишних пробелов.

```
soup.get_text
text = re.sub(r"\s+", " ", text)
text = re.sub(r"\s+([.,!?;:])", r"\1", text)
```

##Подготовка текста
Заголовок + очищенный текст статьи объединяются в full_text. Токенизация через регулярные выражения (только кириллица), лемматизация через pymorphy3.
Использованные признаки, модели и алгоритмы
#BM25 (rank_bm25):

Лемматизация: pymorphy3
Индекс строился по лемматизированному токенизированному full_text
Поиск возвращал top-10 кандидатов(в гибридном 200)

## Dense embeddings

Модель: intfloat/multilingual-e5-large (560M параметров)

Префиксы: "passage:" для статей, "query:" для запросов

Чанкование: статьи длиннее 480 слов разбивались на чанки с перекрытием 100 слов

Max pooling: для статьи брался максимальный скор среди эмбеддингов её чанков

##Гибридный поиск:

Линейная комбинация: alpha * BM25_norm + (1 - alpha) * Dense_norm
Нормализация min-max обоих скоров в [0, 1]

## Как проверял качество на calibration.f
Метрика: MAP@10, самописная реализация. Для каждого запроса считается Average Precision@10, затем усредняется по всем 500 запросам.
Валидация: MAP@10 на calibration — 0.4260, на test — 0.4659.

## Какие типы ошибок нашёл при анализе и что с ними сделал.
Длинные статьи теряли информацию: У 25% статей текст превышает контекстное окно модели в 514 токенов, из-за чего важные фрагменты обрезались. Разбил статьи на чанки по 480 слов с перекрытием 100 слов и брал максимальный скор по чанкам — dense MAP вырос с 0.296 до 0.362.

In [2]:
import pandas as pd
from rank_bm25 import BM25Okapi
from bs4 import BeautifulSoup
import re
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
articles = pd.read_feather("articles.f")
calibration = pd.read_feather("calibration.f")


,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ..."


In [4]:
#Очистим наши запросы от ненужных символов
# Функция очистки
def clean_html(html: str) -> str:
    if html is None:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text.strip()

In [5]:
# Теперь мы можем перейти к токенезации очищенного текста для BM25
def tokenize(text: str) -> list[str]:
    text = text.lower()
    return re.findall(r"\w+", text)

In [31]:
articles['clean_body']=articles['body'].apply(clean_html)
articles["full_text"] = (articles["title"]+' '+articles["title"]+' '+articles["title"]+' ' + articles["clean_body"]) # текст для токенезации заголовок+ текст самой статьи усилил заголовок чтобы BM25 больше на него 'полагался'

In [32]:
#AP@10 = (1 / min(|R_q|, 10)) * sum_i [p_i in R_q] * Precision@i надо реализовать метрику отдельно длелаем AP10 потом делаем MAP100

#AP@10
def average_precision_at_k(predicted: list[str], relevant: set[str], k: int = 10) -> float:
    """
    Расчет AP@K для одного запроса.
    predicted: список предсказанных article_id в порядке ранжирования
    relevant: множество правильных article_id
    k: глубина оценки
    """

    score = 0.0
    hits = 0

    for i, article_id in enumerate(predicted[:k], start=1):

        if article_id in relevant:
            hits += 1
            score += hits / i

    if len(relevant) == 0:
        return 0.0

    return score / min(len(relevant), k)

In [33]:
# очевидно что результат слабый попробую добавить лемматизацию
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

def lemmatize(tokens: list[str]) -> list[str]:
    lemmas = []

    for token in tokens:
        lemma = morph.parse(token)[0].normal_form
        lemmas.append(lemma)

    return lemmas

In [34]:
articles["tokens"]=articles["full_text"].apply(tokenize)
articles["lemmas"] = articles["tokens"].apply(lemmatize)
bm25 = BM25Okapi(articles["lemmas"].tolist())

In [35]:
def search_bm25(query: str, top_k=10) -> list:

    query_tokens = lemmatize(tokenize(query))
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    result = articles.iloc[top_indices][["article_id", "title"]].copy()
    result["score"] = scores[top_indices]
    return result

In [36]:
def map_at_10(calibration, search_func) -> float:
    scores = []

    for _, row in calibration.iterrows():
        query = row["query_text"]
        true_ids = set(row["ground_truth"].split())
        result = search_func(query, top_k=10)  # ← любая функция поиска
        predicted_ids = result["article_id"].astype(str).tolist()
        ap = average_precision_at_k(predicted_ids, true_ids, k=10)
        scores.append(ap)

    return sum(scores) / len(scores)



In [14]:
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda") # Сначала я использовал модель base поэтому тесты будут отличаться
print(model.device)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

cuda:0


In [15]:
def search_embeddings(query: str, top_k: int = 10) -> pd.DataFrame:
    query_embedding = model.encode("query: " + query, normalize_embeddings=True)
    scores = article_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]
    result = articles.iloc[top_indices][["article_id", "title"]].copy()
    result["score"] = scores[top_indices]
    return result

In [17]:
def search_hybrid(query: str, top_k: int = 10, alpha: float = 0.7) -> pd.DataFrame:
    # BM25 скоры для топ-100
    bm25_result = search_bm25(query, top_k=200)
    bm25_scores = bm25_result["score"].values

    # Dense скоры
    query_embedding = model.encode("query: " + query, normalize_embeddings=True)
    dense_scores = article_embeddings[bm25_result.index] @ query_embedding

    # Нормализация min-max в [0, 1]
    bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)
    dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-8)

    # Линейная комбинация
    final_scores = alpha * bm25_norm + (1 - alpha) * dense_norm

    # Ранжируем
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    result = bm25_result.iloc[top_indices][["article_id", "title"]].copy()
    result["score"] = final_scores[top_indices]

    return result

In [37]:
articles["full_text"]="passage: " +articles["full_text"] #  добавил префикс passage:

In [38]:
article_embeddings = model.encode(articles["full_text"].tolist(),
     batch_size=64,
     normalize_embeddings=True,
     show_progress_bar=True)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [22]:
# BM25
map_bm25 = map_at_10(calibration, search_bm25)
print(f"BM25 MAP@10: {map_bm25:.4f}")

# Dense embeddings
map_dense = map_at_10(calibration, search_embeddings)
print(f"Dense MAP@10: {map_dense:.4f}")

# Hybrid с разными alpha
for alpha in [0.20, 0.25, 0.30, 0.35]:
    def search_hybrid_alpha(query, top_k=10):
        return search_hybrid(query, top_k, alpha=alpha)

    map_hybrid = map_at_10(calibration, search_hybrid_alpha)
    print(f"Hybrid (alpha={alpha}) MAP@10: {map_hybrid:.4f}")

BM25 MAP@10: 0.2661


KeyboardInterrupt: 

In [23]:
articles["text_len"] = articles["full_text"].apply(lambda x: len(x.split()))
print(articles["text_len"].describe())

# Топ-10 самых длинных
print("\nТоп-10 самых длинных статей:")
print(articles.nlargest(10, "text_len")[["article_id", "title", "text_len"]])

count      793.000000
mean       737.998739
std       2391.005503
min          6.000000
25%        176.000000
50%        336.000000
75%        700.000000
max      61801.000000
Name: text_len, dtype: float64

Топ-10 самых длинных статей:
     article_id                                              title  text_len
245        2924            Какими товарами интересуются покупатели     61801
730        4436                                    Уровень сервиса     11237
620        4321                                Мой уровень сервиса     11010
527        4220                            Объявление о транспорте      7335
572        4266                                       Недвижимость      7240
549        4243                      Вещи, техника и другие товары      6557
540        4234               Как продавать и покупать с доставкой      5326
606        4307                        Лимит бесплатных размещений      4328
563        4257  Как работать с Защитой сделки: информация для ...    

Тут можно увидеть, что медиана 347 слов, но есть статьи по 60000+ слов

75% перцентиль — 711 слов, что уже превышает контекстное окно e5-large (514 токенов)

Значит ~25% статей обрезаются без чанкования

In [39]:
def chunk_text(text, max_tokens=480, overlap=100):
    """Разбивает текст на чанки с перекрытием"""
    words = text.split()
    chunks = []

    if len(words) <= max_tokens:
        return [text]

    for i in range(0, len(words), max_tokens - overlap):
        chunk = " ".join(words[i:i + max_tokens])
        if chunk:
            chunks.append(chunk)

    return chunks

# Создаём чанки
print("Разбиваем статьи на чанки...")
all_chunks = []
chunk_to_article = []  # маппинг чанк → индекс статьи

for idx, row in articles.iterrows():
    chunks = chunk_text(row["full_text"], max_tokens=480, overlap=100)
    for chunk in chunks:
        all_chunks.append(f"passage: {chunk}")
        chunk_to_article.append(idx)

print(f"Всего чанков: {len(all_chunks)} (было статей: {len(articles)})")

# Эмбеддинги чанков
print("Считаем эмбеддинги чанков...")
chunk_embeddings = model.encode(
    all_chunks,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

Разбиваем статьи на чанки...
Всего чанков: 1901 (было статей: 793)
Считаем эмбеддинги чанков...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

In [40]:
def search_hybrid_chunks(query: str, top_k: int = 10, alpha: float = 0.3) -> pd.DataFrame:
    # Шаг 1: BM25 получает кандидатов
    bm25_candidates = search_bm25(query, top_k=200)
    bm25_scores = bm25_candidates["score"].values

    # Шаг 2: Dense скоры через max pooling чанков
    query_emb = model.encode("query: " + query, normalize_embeddings=True)
    chunk_scores = chunk_embeddings @ query_emb

    # Собираем скоры по статьям
    article_max_score = np.full(len(articles), -1.0)
    article_sum_score = np.zeros(len(articles))
    article_chunk_count = np.zeros(len(articles))

    for chunk_idx, article_idx in enumerate(chunk_to_article):
        score = chunk_scores[chunk_idx]
        if score > article_max_score[article_idx]:
            article_max_score[article_idx] = score
        article_sum_score[article_idx] += score
        article_chunk_count[article_idx] += 1

    # Mean pooling (избегаем деления на 0)
    article_chunk_count = np.maximum(article_chunk_count, 1)
    article_mean_score = article_sum_score / article_chunk_count

    # Комбинируем max и mean
    article_dense_scores = 0.7 * article_max_score + 0.3 * article_mean_score

    # Берём только BM25-кандидатов
    candidate_dense = article_dense_scores[bm25_candidates.index]

    # Нормализация в [0, 1]
    bm25_norm = normalize_scores(bm25_scores)
    dense_norm = normalize_scores(candidate_dense)

    # Взвешенная сумма
    final_scores = alpha * bm25_norm + (1 - alpha) * dense_norm

    # Топ-K
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    return bm25_candidates.iloc[top_indices][["article_id", "title"]]

def normalize_scores(scores):
    """Min-max нормализация в [0, 1]"""
    min_s, max_s = scores.min(), scores.max()
    if max_s == min_s:
        return np.zeros_like(scores)
    return (scores - min_s) / (max_s - min_s)

In [41]:
print("Dense (chunks) only:")
map_dense_chunks = map_at_10(calibration, lambda q, top_k=10: search_hybrid_chunks(q, top_k=top_k, alpha=0.0))
print(f"MAP@10: {map_dense_chunks:.4f}")

print("\nHybrid с чанками:")
for alpha in [0.2]:
    def search_chunks_alpha(query, top_k=10):
        return search_hybrid_chunks(query, top_k=top_k, alpha=alpha)

    map_score = map_at_10(calibration, search_chunks_alpha)
    print(f"alpha={alpha}: MAP@10 = {map_score:.4f}")

Dense (chunks) only:
MAP@10: 0.3620

Hybrid с чанками:
alpha=0.2: MAP@10 = 0.4260


In [42]:
test = pd.read_feather("test.f")
test["answer"] = test["query_text"].apply(lambda q: " ".join(search_hybrid_chunks(q, top_k=10, alpha=0.2)["article_id"].astype(str).tolist()))
test[["query_id", "answer"]].to_csv("answer.csv", index=False)